# `01_run_sam1_pipeline.ipynb` — SAM1 Box-Prompt Data Preparation

A 4-step pipeline executing modules in `src/data_prep_SAM1/` to generate pseudo-ground-truth segmentation masks from bounding-box prompts.

## Directory Layout

```
data/
├── raw/ECUSTFD/                                  # Input VOC annotations & images
└── processed/
    ├── bbox_full/                                # Step 1: Bounding-box verification
    │   ├── images/                               # Bounding box renders
    │   ├── grids/                                # Per-class contact sheets
    │   └── Annotations_patched/                  # Corrected VOC XML files
    └── sam_masks_full/                           # Step 2 & 4: SAM mask generation
        ├── masks/<stem>.npy                      # Serialized mask bundles
        ├── images/<stem>.jpg                     # 3-panel visualizations
        └── grids/
```

## Conventions
- Bounding boxes are read from VOC XMLs (`data/raw/ECUSTFD/Annotations/`), prioritizing `Annotations_patched/` (contains 37 corrected annotations).
- Output `masks/<stem>.npy` stores binary masks for all detected instances per image.


# Step 0 — Download SAM1 (vit_b) Checkpoint

Downloads `sam_vit_b_01ec64.pth` (~375 MB) from Facebook AI Research repository if not already cached in `models/sam/`.


In [ ]:
# Cell 0 — Download SAM1 vit_b checkpoint if missing
import urllib.request
from pathlib import Path

PROJ = Path("E:/AI_Research/dlt8")
SAM_DIR = PROJ / "models" / "sam"
SAM_CKPT = SAM_DIR / "sam_vit_b_01ec64.pth"
SAM_DIR.mkdir(parents=True, exist_ok=True)

if SAM_CKPT.exists():
    size_mb = SAM_CKPT.stat().st_size / (1024 * 1024)
    print("[OK] SAM checkpoint exists: %s (%.1f MB)" % (SAM_CKPT, size_mb))
else:
    print("[DL] downloading SAM checkpoint to: %s" % SAM_CKPT)
    url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
    tmp = SAM_CKPT.with_suffix(".pth.tmp")
    urllib.request.urlretrieve(url, tmp)
    tmp.replace(SAM_CKPT)
    size_mb = SAM_CKPT.stat().st_size / (1024 * 1024)
    print("[OK] downloaded: %s (%.1f MB)" % (SAM_CKPT, size_mb))


# Step 1 — Visualize Bounding Boxes (Including Patched XMLs)

Runs `src/data_prep_SAM1/visual_bbox/visualize_bbox_full.py` to inspect bounding boxes and generate visual contact sheets per class.


In [3]:
# Cell 1 — Run visualize_bbox_full.py --clean
#
# Helper `run_script_step(...)`:
#   - chạy subprocess, stream từng dòng stdout ra console của cell này
#     (lưu vào lịch sử notebook) ĐỒNG THỜI ghi ra file log riêng
#     `data/processed/<step>/logs/run_<YYYYMMDD_HHMMSS>.log` để xem lại
#   - mỗi lần chạy tạo file log mới, KHÔNG overwrite log cũ
#   - tham số `step_tag` chỉ dùng để đặt tên thư mục logs/ cho dễ tra cứu
import subprocess, sys, io
from pathlib import Path
from datetime import datetime

PROJ = Path("E:/AI_Research/dlt8")

def run_script_step(rel_script: str, args: list[str], step_tag: str) -> int:
    """Run a python script with live streaming + per-run log file.

    Returns the process exit code.
    """
    script = PROJ / rel_script
    log_dir = PROJ / "data" / "processed" / step_tag / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = log_dir / f"run_{ts}.log"

    header = (
        f"[run] {script.name} {' '.join(args)}\n"
        f"[cwd] {PROJ}\n"
        f"[log] {log_file}\n"
        f"[ts ] {datetime.now().isoformat(timespec='seconds')}\n"
        f"{'-' * 60}\n"
    )
    print(header, end="")
    with log_file.open("w", encoding="utf-8") as lf:
        lf.write(header)
        # Stream line-by-line so Jupyter shows progress AND file gets a copy
        proc = subprocess.Popen(
            [sys.executable, str(script), *args],
            cwd=str(PROJ),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            encoding="utf-8",
            errors="replace",
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            lf.write(line)
        rc = proc.wait()
        footer = f"{'-' * 60}\n[exit] {rc}\n"
        print(footer, end="")
        lf.write(footer)
    return rc

rc = run_script_step(
    "src/data_prep_SAM1/visual_bbox/visualize_bbox_full.py",
    ["--clean"],
    step_tag="bbox_full",
)
print(f"[exit] {rc}")


[run] visualize_bbox_full.py --clean
[cwd] E:\AI_Research\dlt8
[log] E:\AI_Research\dlt8\data\processed\bbox_full\logs\run_20260801_225314.log
[ts ] 2026-08-01T22:53:14
------------------------------------------------------------
[scan] XML annotations: 2978 | invalid: 0
[scan] images: 2978 | missing: 0
[scan] annotated objects: 6062
  apple: 322 objects
  saved 00_apple_p01_of_05.png (80 tiles)
  saved 00_apple_p02_of_05.png (80 tiles)
  saved 00_apple_p03_of_05.png (80 tiles)
  saved 00_apple_p04_of_05.png (80 tiles)
  saved 00_apple_p05_of_05.png (2 tiles)
  banana: 212 objects
  saved 01_banana_p01_of_03.png (80 tiles)
  saved 01_banana_p02_of_03.png (80 tiles)
  saved 01_banana_p03_of_03.png (52 tiles)
  bread: 66 objects
  saved 02_bread_p01_of_01.png (66 tiles)
  bun: 90 objects
  saved 03_bun_p01_of_02.png (80 tiles)
  saved 03_bun_p02_of_02.png (10 tiles)
  coin: 2976 objects
  saved 04_coin_p01_of_38.png (80 tiles)
  saved 04_coin_p02_of_38.png (80 tiles)
  saved 04_coin_p03_

# Step 2 — Segment ECUSTFD via SAM1 Box-Prompting

Runs `src/data_prep_SAM1/sam_masks_full/segment_sam1_box.py` using GT bounding boxes as prompts to generate instance masks for all 2,978 images (~19 minutes on RTX 4050).


In [6]:
# Cell 2 — Run segment_sam1_box.py --clean --workers 1
#
# Helper `run_script_step(...)` chạy subprocess, stream từng dòng stdout
# ra console của cell (lưu vào lịch sử notebook) ĐỒNG THỜI ghi ra file
# log riêng `data/processed/sam_masks_full/logs/run_<timestamp>.log` để
# xem lại. Mỗi lần chạy tạo file log mới, KHÔNG overwrite log cũ.
import subprocess, sys
from pathlib import Path
from datetime import datetime

PROJ = Path("E:/AI_Research/dlt8")

def run_script_step(rel_script: str, args: list[str], step_tag: str) -> int:
    script = PROJ / rel_script
    log_dir = PROJ / "data" / "processed" / step_tag / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = log_dir / f"run_{ts}.log"
    header = (
        f"[run] {script.name} {' '.join(args)}\n"
        f"[cwd] {PROJ}\n"
        f"[log] {log_file}\n"
        f"[ts ] {datetime.now().isoformat(timespec='seconds')}\n"
        f"{'-' * 60}\n"
    )
    print(header, end="")
    with log_file.open("w", encoding="utf-8") as lf:
        lf.write(header)
        proc = subprocess.Popen(
            [sys.executable, str(script), *args],
            cwd=str(PROJ),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            encoding="utf-8",
            errors="replace",
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            lf.write(line)
        rc = proc.wait()
        footer = f"{'-' * 60}\n[exit] {rc}\n"
        print(footer, end="")
        lf.write(footer)
    return rc

rc = run_script_step(
    "src/data_prep_SAM1/sam_masks_full/segment_sam1_box.py",
    ["--clean", "--workers", "1"],
    step_tag="sam_masks_full",
)
print(f"[exit] {rc}")


[run] segment_sam1_box.py --clean --workers 1
[cwd] E:\AI_Research\dlt8
[log] E:\AI_Research\dlt8\data\processed\sam_masks_full\logs\run_20260801_235834.log
[ts ] 2026-08-01T23:58:34
------------------------------------------------------------
[scan] xml stems: 2978  workers: 1  device: cuda
[scan] patched source: 37  raw source: 2941
  [   1/2978] apple001S(1)             src=raw_gt_box_prompt                obj=2 seg=2
  [   2/2978] apple001S(2)             src=raw_gt_box_prompt                obj=2 seg=2
  [   3/2978] apple001T(1)             src=raw_gt_box_prompt                obj=2 seg=2
  [   4/2978] apple001T(2)             src=raw_gt_box_prompt                obj=2 seg=2
  [   5/2978] apple002S(1)             src=raw_gt_box_prompt                obj=2 seg=2
  [   6/2978] apple002S(2)             src=raw_gt_box_prompt                obj=2 seg=2
  [   7/2978] apple002S(3)             src=raw_gt_box_prompt                obj=2 seg=2
  [   8/2978] apple002S(4)             src=raw_

# Step 3 — Apply Manual Mask Overrides

Applies manual mask corrections from `data/annotation/manual_masks/` via `src/data_prep_SAM1/apply_masks/apply_mask_overrides.py` for edge cases where SAM leakage occurs.


In [7]:
# Cell 3 — Run apply_mask_overrides.py (in-place)
#
# Helper `run_script_step(...)` chạy subprocess, stream từng dòng stdout
# ra console của cell (lưu vào lịch sử notebook) ĐỒNG THỜI ghi ra file
# log riêng `data/processed/sam_masks_full/logs/run_<timestamp>.log` để
# xem lại. Mỗi lần chạy tạo file log mới, KHÔNG overwrite log cũ.
import subprocess, sys
from pathlib import Path
from datetime import datetime

PROJ = Path("E:/AI_Research/dlt8")

def run_script_step(rel_script: str, args: list[str], step_tag: str) -> int:
    script = PROJ / rel_script
    log_dir = PROJ / "data" / "processed" / step_tag / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = log_dir / f"run_{ts}.log"
    header = (
        f"[run] {script.name} {' '.join(args)}\n"
        f"[cwd] {PROJ}\n"
        f"[log] {log_file}\n"
        f"[ts ] {datetime.now().isoformat(timespec='seconds')}\n"
        f"{'-' * 60}\n"
    )
    print(header, end="")
    with log_file.open("w", encoding="utf-8") as lf:
        lf.write(header)
        proc = subprocess.Popen(
            [sys.executable, str(script), *args],
            cwd=str(PROJ),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            encoding="utf-8",
            errors="replace",
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            lf.write(line)
        rc = proc.wait()
        footer = f"{'-' * 60}\n[exit] {rc}\n"
        print(footer, end="")
        lf.write(footer)
    return rc

rc = run_script_step(
    "src/data_prep_SAM1/apply_masks/apply_mask_overrides.py",
    [],
    step_tag="sam_masks_full",
)
print(f"[exit] {rc}")


[run] apply_mask_overrides.py 
[cwd] E:\AI_Research\dlt8
[log] E:\AI_Research\dlt8\data\processed\sam_masks_full\logs\run_20260802_003903.log
[ts ] 2026-08-02T00:39:03
------------------------------------------------------------
== apply_mask_overrides.py | mode=WRITE-IN-PLACE -> E:\AI_Research\dlt8\data\processed\sam_masks_full\masks | n=1 ==
[OK ] grape001T(5)  class=grape  obj_idx=0  area: 9284 -> 118862  bbox_voc: [262, 4, 686, 446] -> [262, 4, 686, 446]  src: raw_gt_box_prompt -> manual_mask_override  png=data/annotation/manual_masks/grape001T(5)_grape.png  // SAM bbox quá rộng, mask bám background. Thay mask thủ công.  // wrote in-place  sha256: 746d64a754c3.. -> 0b0a9b533a56..
---
Summary: ok=1  skip=0

[next] regenerate images / grids / CSVs:
  python src/data_prep_SAM1/sam_masks_full/visualize_results.py --clean

------------------------------------------------------------
[exit] 0
[exit] 0


# Step 4 — Visualize SAM Masks (3-Panel Overlays & Summary Grids)

Runs `src/data_prep_SAM1/sam_masks_full/visualize_results.py` to produce 3-panel visualizations `[original | +bbox | +mask]` and evaluation reports (`_segmentation_report.md`, `_poor_miou_report.csv`).


In [8]:
# Cell 4 — Run visualize_results.py --clean
#
# Helper `run_script_step(...)` chạy subprocess, stream từng dòng stdout
# ra console của cell (lưu vào lịch sử notebook) ĐỒNG THỜI ghi ra file
# log riêng `data/processed/sam_masks_full/logs/run_<timestamp>.log` để
# xem lại. Mỗi lần chạy tạo file log mới, KHÔNG overwrite log cũ.
import subprocess, sys
from pathlib import Path
from datetime import datetime

PROJ = Path("E:/AI_Research/dlt8")

def run_script_step(rel_script: str, args: list[str], step_tag: str) -> int:
    script = PROJ / rel_script
    log_dir = PROJ / "data" / "processed" / step_tag / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = log_dir / f"run_{ts}.log"
    header = (
        f"[run] {script.name} {' '.join(args)}\n"
        f"[cwd] {PROJ}\n"
        f"[log] {log_file}\n"
        f"[ts ] {datetime.now().isoformat(timespec='seconds')}\n"
        f"{'-' * 60}\n"
    )
    print(header, end="")
    with log_file.open("w", encoding="utf-8") as lf:
        lf.write(header)
        proc = subprocess.Popen(
            [sys.executable, str(script), *args],
            cwd=str(PROJ),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            encoding="utf-8",
            errors="replace",
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            lf.write(line)
        rc = proc.wait()
        footer = f"{'-' * 60}\n[exit] {rc}\n"
        print(footer, end="")
        lf.write(footer)
    return rc

rc = run_script_step(
    "src/data_prep_SAM1/sam_masks_full/visualize_results.py",
    ["--clean"],
    step_tag="sam_masks_full",
)
print(f"[exit] {rc}")


The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.[run] visualize_results.py --clean
[cwd] E:\AI_Research\dlt8
[log] E:\AI_Research\dlt8\data\processed\sam_masks_full\logs\run_20260802_003918.log
[ts ] 2026-08-02T00:39:18
------------------------------------------------------------

[scan] masks: 2978  workers: 8  miou_thr: 0.5
  render 100/2978  elapsed=2.1s  rate=47.45/s
  render 200/2978  elapsed=3.2s  rate=63.34/s
  render 300/2978  elapsed=4.0s  rate=74.12/s
  render 400/2978  elapsed=5.0s  rate=79.41/s
  render 500/2978  elapsed=6.0s  rate=83.31/s
  render 600/2978  elapsed=7.1s  rate=84.84/s
  render 700/2978  elapsed=8.3s  rate=84.18/s
  render 800/2978  elapsed=9.3s  rate=85.62/s
  render 900/2978  elapsed=10.4s  rate=86.81/s
  render 1000/2978  elapsed=11.4s  rate=88.09/s
  render 1100/2978  elapsed=12.2s  rate=90.03/s
  render 1200/2978  elapsed=13.2s  rate=90.61/s
  render 1300/2978  

# Pipeline Completion

Output artifacts:
- `data/processed/sam_masks_full/masks/<stem>.npy` (2,978 mask files)
- `data/processed/sam_masks_full/_poor_miou_report.csv`
- `data/processed/sam_masks_full/_segmentation_report.md`

Proceed to YOLO dataset conversion in `02_yolo_seg_pipeline.ipynb`.
